# Nettoyage et Preparation des Donnees adaptés au deep

## Prudential Life Insurance Assessment

**Objectif** : Predire le niveau de risque (`Response`, variable ordinale a 8 niveaux) des candidats a une assurance vie a partir de plus de 100 variables decrivant leurs profils.  
**Source** : [Kaggle](https://www.kaggle.com/c/prudential-life-insurance-assessment)

---

### Plan

1. Chargement des donnees
2. Suppression des colonnes avec >95% de valeurs manquantes (avec verification)
3. Imputation des valeurs manquantes
4. Encodage des variables textuelles (Label Encoding)
5. Target Encoding pour les colonnes a haute cardinalite
6. Feature Engineering
7. Sauvegarde

---
## Installation des dependances

In [1]:
!pip install pandas numpy scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\coche\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Imports et Configuration

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
import warnings
import os

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

### Definition des types de variables (selon la documentation Kaggle)

In [3]:
CATEGORICAL_COLS = [
    "Product_Info_1", "Product_Info_2", "Product_Info_3", "Product_Info_5",
    "Product_Info_6", "Product_Info_7",
    "Employment_Info_2", "Employment_Info_3", "Employment_Info_5",
    "InsuredInfo_1", "InsuredInfo_2", "InsuredInfo_3", "InsuredInfo_4",
    "InsuredInfo_5", "InsuredInfo_6", "InsuredInfo_7",
    "Insurance_History_1", "Insurance_History_2", "Insurance_History_3",
    "Insurance_History_4", "Insurance_History_7", "Insurance_History_8",
    "Insurance_History_9",
    "Family_Hist_1",
    "Medical_History_2", "Medical_History_3", "Medical_History_4",
    "Medical_History_5", "Medical_History_6", "Medical_History_7",
    "Medical_History_8", "Medical_History_9", "Medical_History_11",
    "Medical_History_12", "Medical_History_13", "Medical_History_14",
    "Medical_History_16", "Medical_History_17", "Medical_History_18",
    "Medical_History_19", "Medical_History_20", "Medical_History_21",
    "Medical_History_22", "Medical_History_23", "Medical_History_25",
    "Medical_History_26", "Medical_History_27", "Medical_History_28",
    "Medical_History_29", "Medical_History_30", "Medical_History_31",
    "Medical_History_33", "Medical_History_34", "Medical_History_35",
    "Medical_History_36", "Medical_History_37", "Medical_History_38",
    "Medical_History_39", "Medical_History_40", "Medical_History_41",
]

CONTINUOUS_COLS = [
    "Product_Info_4", "Ins_Age", "Ht", "Wt", "BMI",
    "Employment_Info_1", "Employment_Info_4", "Employment_Info_6",
    "Insurance_History_5",
    "Family_Hist_2", "Family_Hist_3", "Family_Hist_4", "Family_Hist_5",
]

DISCRETE_COLS = [
    "Medical_History_1", "Medical_History_10", "Medical_History_15",
    "Medical_History_24", "Medical_History_32",
]

DUMMY_COLS = [f"Medical_Keyword_{i}" for i in range(1, 49)]

TARGET = "Response"

---
## 1. Chargement des donnees

In [5]:
train = pd.read_csv("data/train.csv")
test  = pd.read_csv("data/test.csv")

print(f"Train : {train.shape[0]:,} lignes x {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes x {test.shape[1]} colonnes")

# Combiner train + test pour des transformations coherentes
train["is_train"] = 1
test["is_train"]  = 0
if TARGET not in test.columns:
    test[TARGET] = np.nan

df = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Jeu combine : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Train : 59,381 lignes x 128 colonnes
Test  : 19,765 lignes x 127 colonnes
Jeu combine : 79,146 lignes x 129 colonnes


---
## 2. Suppression des colonnes avec >95% de valeurs manquantes

Avant de supprimer, on verifie si les quelques valeurs presentes dans ces colonnes
ont un lien significatif avec la variable cible `Response`.  
Si dans les 5% restants il y a un signal fort, la colonne est precieuse malgre le vide.

In [6]:
MISSING_THRESHOLD = 0.95

feature_cols = [c for c in df.columns if c not in ["Id", "is_train", TARGET]]
missing_pct = (df[feature_cols].isnull().sum() / len(df) * 100).round(2)

candidates_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD * 100].index.tolist()

print(f"Colonnes candidates a la suppression (>{MISSING_THRESHOLD*100:.0f}% manquant) : {len(candidates_to_drop)}")
for c in candidates_to_drop:
    print(f"   - {c} ({missing_pct[c]:.1f}% manquant)")

Colonnes candidates a la suppression (>95% manquant) : 2
   - Medical_History_10 (99.0% manquant)
   - Medical_History_32 (98.2% manquant)


In [7]:
# Verification : est-ce que les valeurs non-manquantes sont correlees a Response ?
# On utilise uniquement le train (le test n'a pas de Response)
df_train = df[df["is_train"] == 1].copy()
global_mean_response = df_train[TARGET].mean()

print(f"Moyenne globale de Response : {global_mean_response:.3f}")
print(f"{'─' * 80}")
print(f"{'Colonne':<25} {'% NaN':>8} {'N present':>10} {'Moy Response (present)':>25} {'Moy Response (absent)':>25} {'Ecart':>8}")
print(f"{'─' * 80}")

cols_to_drop = []
cols_to_keep = []

for col in candidates_to_drop:
    present_mask = df_train[col].notna()
    n_present = present_mask.sum()
    
    if n_present < 10:
        # Trop peu de donnees pour juger -> supprimer
        cols_to_drop.append(col)
        print(f"{col:<25} {missing_pct[col]:>7.1f}% {n_present:>10} {'trop peu de donnees':>25} {'':>25} {'DROP':>8}")
        continue
    
    mean_present = df_train.loc[present_mask, TARGET].mean()
    mean_absent  = df_train.loc[~present_mask, TARGET].mean()
    ecart = abs(mean_present - mean_absent)
    
    # Si l'ecart entre present/absent est > 1 point de Response, c'est un signal
    if ecart > 1.0:
        decision = "KEEP"
        cols_to_keep.append(col)
    else:
        decision = "DROP"
        cols_to_drop.append(col)
    
    print(f"{col:<25} {missing_pct[col]:>7.1f}% {n_present:>10} {mean_present:>25.3f} {mean_absent:>25.3f} {decision:>8}")

print(f"\n{'─' * 80}")
print(f"Resultat : {len(cols_to_drop)} colonnes a supprimer, {len(cols_to_keep)} colonnes a garder")
if cols_to_keep:
    print(f"Colonnes gardees malgre >95% NaN (signal fort) : {cols_to_keep}")

Moyenne globale de Response : 5.637
────────────────────────────────────────────────────────────────────────────────
Colonne                      % NaN  N present    Moy Response (present)     Moy Response (absent)    Ecart
────────────────────────────────────────────────────────────────────────────────
Medical_History_10           99.0%        557                     4.616                     5.647     KEEP
Medical_History_32           98.2%       1107                     4.522                     5.658     KEEP

────────────────────────────────────────────────────────────────────────────────
Resultat : 0 colonnes a supprimer, 2 colonnes a garder
Colonnes gardees malgre >95% NaN (signal fort) : ['Medical_History_10', 'Medical_History_32']


In [8]:
# Suppression uniquement des colonnes sans signal
df.drop(columns=cols_to_drop, inplace=True)

CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c not in cols_to_drop]
CONTINUOUS_COLS  = [c for c in CONTINUOUS_COLS if c not in cols_to_drop]
DISCRETE_COLS    = [c for c in DISCRETE_COLS if c not in cols_to_drop]
DUMMY_COLS       = [c for c in DUMMY_COLS if c not in cols_to_drop]

print(f"{len(cols_to_drop)} colonnes supprimees.")
print(f"Dimensions apres suppression : {df.shape}")

0 colonnes supprimees.
Dimensions apres suppression : (79146, 129)


---
## 3. Imputation des valeurs manquantes

| Type | Strategie | Justification |
|:-----|:----------|:--------------|
| Continue (BMI, Age...) | Mediane | Robuste aux valeurs extremes |
| Discrete (Medical History) | -1 | L'absence d'info est un signal en soi |
| Categorielle (codes medicaux) | Missing | plus adpapté que -1 pour deep |

In [9]:
# Variables continues -> Mediane
continuous_present = [c for c in CONTINUOUS_COLS if c in df.columns]
for col in continuous_present:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

# Variables discretes -> -1
discrete_present = [c for c in DISCRETE_COLS if c in df.columns]
for col in discrete_present:
    if df[col].isnull().any():
        df[col].fillna(-1, inplace=True)

# Variables categorielles -> "Missing"
categorical_present = [c for c in CATEGORICAL_COLS if c in df.columns]
for col in categorical_present:
    if df[col].isnull().any():
        df[col].fillna("Missing", inplace=True)

# Imputation de securite pour tout ce qui reste
remaining_features = [c for c in df.columns if c not in ["Id", "is_train", TARGET]]
for col in remaining_features:
    if df[col].isnull().any():
        if df[col].dtype == "object":
            df[col].fillna("Missing", inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

nan_count = df[remaining_features].isnull().sum().sum()
print(f"Valeurs manquantes restantes : {nan_count}")

Valeurs manquantes restantes : 0


---
## 4. Encodage des variables textuelles (Label Encoding)

La colonne `Product_Info_2` contient des codes textuels ("A1", "D3"...). On les transforme en entiers.

In [10]:
text_cols = df[remaining_features].select_dtypes(include=["object"]).columns.tolist()

for col in text_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"{col} : {len(le.classes_)} categories -> [{df[col].min()}, {df[col].max()}]")

if not text_cols:
    print("Aucune colonne textuelle a encoder.")

Product_Info_2 : 19 categories -> [0, 18]


---
## 5. Target Encoding pour les colonnes a haute cardinalite

On remplace chaque categorie par la moyenne de `Response` pour cette categorie.  
K-Fold (5 folds) pour eviter le data leakage.

On ne le fait pas ici, car ce n’est pas adapté à la génération de données synthétiques. Le target encoding utilise directement la variable cible Response pour transformer certaines variables explicatives, ce qui modifie artificiellement la structure réelle des données. Cela peut être utile plus tard pour entraîner un modèle prédictif final, mais pas à cette étape de préparation du dataset.

---
## 6. Feature Engineering

In [11]:
med_kw_cols = [c for c in DUMMY_COLS if c in df.columns]
if med_kw_cols:
    df["Medical_Keyword_Count"] = df[med_kw_cols].sum(axis=1)

if "BMI" in df.columns and "Ins_Age" in df.columns:
    df["BMI_Age"] = df["BMI"] * df["Ins_Age"]

if "Wt" in df.columns and "Ht" in df.columns:
    df["Wt_Ht_ratio"] = df["Wt"] / (df["Ht"] + 1e-8)

print("Features ajoutees : Medical_Keyword_Count, BMI_Age, Wt_Ht_ratio")

Features ajoutees : Medical_Keyword_Count, BMI_Age, Wt_Ht_ratio


---
## 7. Sauvegarde

In [12]:
train_clean = df[df["is_train"] == 1].drop(columns=["is_train"])
test_clean  = df[df["is_train"] == 0].drop(columns=["is_train", TARGET])

feature_cols_final = [c for c in train_clean.columns if c not in ["Id", TARGET]]
train_nan = train_clean[feature_cols_final].isnull().sum().sum()
test_nan  = test_clean.drop(columns=["Id"]).isnull().sum().sum()

print(f"Train : {train_clean.shape[0]:,} x {train_clean.shape[1]} | NaN : {train_nan}")
print(f"Test  : {test_clean.shape[0]:,} x {test_clean.shape[1]} | NaN : {test_nan}")
print(f"Features : {len(feature_cols_final)}")

Train : 59,381 x 131 | NaN : 0
Test  : 19,765 x 130 | NaN : 0
Features : 129


In [14]:
OUTPUT_DIR = "data_clean"
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_output = os.path.join(OUTPUT_DIR, "train_clean.csv")
test_output  = os.path.join(OUTPUT_DIR, "test_clean.csv")

train_clean.to_csv(train_output, index=False)
test_clean.to_csv(test_output, index=False)

print(f"Sauvegardes : {train_output}, {test_output}")

Sauvegardes : data_clean\train_clean.csv, data_clean\test_clean.csv


In [15]:
train_clean.head()

,Id,Product_Info_1,Product_Info_2,Product_Info_3,Product_Info_4,Product_Info_5,Product_Info_6,Product_Info_7,Ins_Age,Ht,Wt,BMI,Employment_Info_1,Employment_Info_2,Employment_Info_3,Employment_Info_4,Employment_Info_5,Employment_Info_6,InsuredInfo_1,InsuredInfo_2,InsuredInfo_3,InsuredInfo_4,InsuredInfo_5,InsuredInfo_6,InsuredInfo_7,...,Medical_Keyword_28,Medical_Keyword_29,Medical_Keyword_30,Medical_Keyword_31,Medical_Keyword_32,Medical_Keyword_33,Medical_Keyword_34,Medical_Keyword_35,Medical_Keyword_36,Medical_Keyword_37,Medical_Keyword_38,Medical_Keyword_39,Medical_Keyword_40,Medical_Keyword_41,Medical_Keyword_42,Medical_Keyword_43,Medical_Keyword_44,Medical_Keyword_45,Medical_Keyword_46,Medical_Keyword_47,Medical_Keyword_48,Response,Medical_Keyword_Count,BMI_Age,Wt_Ht_ratio
0,2,1,16,10,0.076923,2,1,1,0.641791,0.581818,0.148536,0.323008,0.028,12,1,0.0,3,0.2500,1,2,6,3,1,2,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8.0,0,0.207304,0.255295
1,5,1,0,26,0.076923,2,3,1,0.059701,0.600000,0.131799,0.272288,0.000,1,3,0.0,2,0.0018,1,2,6,3,1,2,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4.0,0,0.016256,0.219665
2,6,1,18,26,0.076923,2,3,1,0.029851,0.745455,0.288703,0.428780,0.030,9,1,0.0,2,0.0300,1,2,8,3,1,1,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8.0,0,0.012799,0.387284
3,7,1,17,10,0.487179,2,3,1,0.164179,0.672727,0.205021,0.352438,0.042,9,1,0.0,3,0.2000,2,2,8,3,1,2,1,...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8.0,1,0.057863,0.304761
4,8,1,15,26,0.230769,2,3,1,0.417910,0.654545,0.234310,0.424046,0.027,9,1,0.0,2,0.0500,1,2,6,3,1,2,1,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,8.0,0,0.177213,0.357973


---
## 8. Vérification

In [16]:
import pandas as pd

df = pd.read_csv("data_clean/train_clean.csv")

print(df.shape)
print(df.dtypes)
print(df.isnull().sum().sum())
print(df.head())

(59381, 131)
Id                         int64
Product_Info_1             int64
Product_Info_2             int64
Product_Info_3             int64
Product_Info_4           float64
                          ...   
Medical_Keyword_48         int64
Response                 float64
Medical_Keyword_Count      int64
BMI_Age                  float64
Wt_Ht_ratio              float64
Length: 131, dtype: object
0
   Id  Product_Info_1  Product_Info_2  Product_Info_3  Product_Info_4  \
0   2               1              16              10        0.076923   
1   5               1               0              26        0.076923   
2   6               1              18              26        0.076923   
3   7               1              17              10        0.487179   
4   8               1              15              26        0.230769   

   Product_Info_5  Product_Info_6  Product_Info_7   Ins_Age        Ht  \
0               2               1               1  0.641791  0.581818   
1         